### Comparative analysis of automatic time interval vs manual 

##### Manual analysis -> data extraction from .TextGrid files

In [10]:
def extract_time_intervals(file_path):
    """
    Extract time intervals (start and end times) from a TextGrid file.

    This function assumes that the file follows the format where each interval is defined as:
        intervals [n]:
            xmin = <start time>
            xmax = <end time>
            text = "<label>"

    Parameters:
        file_path (str): Path to the TextGrid file.

    Returns:
        list of dict: Each dictionary contains 'start' and 'end' keys with corresponding float values.
    """
    intervals = []
    in_interval = False  # Flag to indicate that we are inside an interval block
    current_interval = {}

    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                
                # When an interval block starts:
                if line.startswith("intervals ["):
                    in_interval = True
                    current_interval = {}

                # Process lines only when inside an interval block
                elif in_interval:
                    if line.startswith("xmin ="):
                        try:
                            current_interval["start"] = float(line.split("=")[1].strip())
                        except ValueError:
                            print("Could not convert start time to float:", line)
                    elif line.startswith("xmax ="):
                        try:
                            current_interval["end"] = float(line.split("=")[1].strip())
                        except ValueError:
                            print("Could not convert end time to float:", line)
                        # Once we have both xmin and xmax, store the interval and reset the flag.
                        if "start" in current_interval and "end" in current_interval:
                            intervals.append(current_interval)
                        in_interval = False  # End of this interval block
        return intervals

    except Exception as e:
        print(f"Error reading file: {e}")
        return []

# Example usage:
if __name__ == "__main__":
    textgrid_file = r"C:\Users\Teresa\Desktop\MBBAS (2ºano)\Tese\Audio_samples\GC-Recortes\GC-F5-3ml-R.TextGrid"  
    time_intervals = extract_time_intervals(textgrid_file)

    if time_intervals:
        print("Extracted time intervals:")
        for interval in time_intervals:
            print(f"Start: {interval['start']}, End: {interval['end']}")
    else:
        print("No time intervals were extracted.")


Extracted time intervals:
Start: 0.0, End: 0.3284604225242112
Start: 0.3284604225242112, End: 0.4637857114964074
Start: 0.4637857114964074, End: 0.46613919478288035
Start: 0.46613919478288035, End: 0.6175


In [9]:
import os
import glob
import csv

def extract_time_intervals(file_path):
    """
    Extract time intervals (start and end times) from a TextGrid file.
    Assumes that intervals are defined as:
    
        intervals [n]:
            xmin = <start time>
            xmax = <end time>
            text = "<label>"

    Parameters:
        file_path (str): Path to the TextGrid file.
    
    Returns:
        list of dict: Each dictionary contains 'start' and 'end' keys with float values.
    """
    intervals = []
    in_interval = False  # Flag indicating we're inside an interval block
    current_interval = {}

    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                
                # Detect start of an interval block.
                if line.startswith("intervals ["):
                    in_interval = True
                    current_interval = {}  # Reset for new interval
                
                # Process lines only when inside an interval block.
                elif in_interval:
                    if line.startswith("xmin ="):
                        try:
                            current_interval["start"] = float(line.split("=")[1].strip())
                        except ValueError:
                            print(f"Could not convert start time to float in file {file_path}: {line}")
                    elif line.startswith("xmax ="):
                        try:
                            current_interval["end"] = float(line.split("=")[1].strip())
                        except ValueError:
                            print(f"Could not convert end time to float in file {file_path}: {line}")
                        # Once both start and end are captured, save the interval.
                        if "start" in current_interval and "end" in current_interval:
                            intervals.append(current_interval)
                        in_interval = False  # Exit the current interval block
        return intervals

    except Exception as e:
        print(f"Error reading file {file_path}: {e}")
        return []


def process_textgrid_files(directory, output_csv):
    """
    Process all TextGrid files in the specified directory, extract time intervals,
    and write the results to a CSV file.

    Parameters:
        directory (str): Path to the directory containing TextGrid files.
        output_csv (str): Path for the output CSV file.
    """
    # Adjust the pattern if your files use a different extension (e.g., ".textgrid")
    pattern = os.path.join(directory, "*.TextGrid")
    files = glob.glob(pattern)
    
    if not files:
        print("No TextGrid files found in the directory.")
        return

    all_intervals = []  # List to store intervals from all files

    for file_path in files:
        intervals = extract_time_intervals(file_path)
        # Add the file name to each interval dictionary for reference.
        for interval in intervals:
            interval["file"] = os.path.basename(file_path)
            all_intervals.append(interval)
    
    if all_intervals:
        # Write the intervals to a CSV file.
        with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['file', 'start', 'end']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for interval in all_intervals:
                writer.writerow(interval)
        print(f"Data successfully written to {output_csv}")
    else:
        print("No intervals were extracted from the files.")


if __name__ == "__main__":
    # Specify the directory containing your TextGrid files.
    textgrid_directory = r"C:\Users\Teresa\Desktop\MBBAS (2ºano)\Tese\Audio_samples\GC-Recortes"
    
    # Specify the path for the output CSV file.
    output_csv_path = r"C:\Users\Teresa\Desktop\MBBAS (2ºano)\Tese\extracted_intervals.csv"
    
    process_textgrid_files(textgrid_directory, output_csv_path)


Data successfully written to C:\Users\Teresa\Desktop\MBBAS (2ºano)\Tese\extracted_intervals.csv


In [10]:
import pandas as pd

csv_file_path = r"C:\Users\Teresa\Desktop\MBBAS (2ºano)\Tese\extracted_intervals.csv"
df = pd.read_csv(csv_file_path)
print(df.head())  # Prints the first 5 rows by default


                    file     start       end
0  GC-F5-10ml-R.TextGrid  0.000000  0.001267
1  GC-F5-10ml-R.TextGrid  0.001267  0.053391
2  GC-F5-10ml-R.TextGrid  0.053391  0.357696
3  GC-F5-10ml-R.TextGrid  0.357696  0.570164
4  GC-F5-10ml-R.TextGrid  0.570164  0.574206


In [23]:
# Load the CSV file (replace 'your_file.csv' with the actual file path)
csv_file_path = r'C:\Users\Teresa\Desktop\MBBAS (2ºano)\Tese\extracted_intervals.csv'
df = pd.read_csv(csv_file_path)

# Calculate the duration (end - start)
df['duration'] = df['end'] - df['start']

df.to_csv(csv_file_path, index=False)
# Display the table
print(df)

                     file     start       end  duration
0   GC-F5-10ml-R.TextGrid  0.000000  0.001267  0.001267
1   GC-F5-10ml-R.TextGrid  0.001267  0.053391  0.052124
2   GC-F5-10ml-R.TextGrid  0.053391  0.357696  0.304305
3   GC-F5-10ml-R.TextGrid  0.357696  0.570164  0.212467
4   GC-F5-10ml-R.TextGrid  0.570164  0.574206  0.004042
5   GC-F5-10ml-R.TextGrid  0.574206  1.060500  0.486294
6   GC-F5-20ml-R.TextGrid  0.000000  0.073245  0.073245
7   GC-F5-20ml-R.TextGrid  0.073245  0.074677  0.001432
8   GC-F5-20ml-R.TextGrid  0.074677  0.406954  0.332277
9   GC-F5-20ml-R.TextGrid  0.406954  0.409819  0.002864
10  GC-F5-20ml-R.TextGrid  0.409819  0.738000  0.328181
11   GC-F5-3ml-R.TextGrid  0.000000  0.328460  0.328460
12   GC-F5-3ml-R.TextGrid  0.328460  0.463786  0.135325
13   GC-F5-3ml-R.TextGrid  0.463786  0.466139  0.002353
14   GC-F5-3ml-R.TextGrid  0.466139  0.617500  0.151361
15   GC-F5-5ml-R.TextGrid  0.000000  0.000120  0.000120
16   GC-F5-5ml-R.TextGrid  0.000120  0.238552  0

In [ ]:
def reduce_intervals(df, target_intervals=3):
    """Merge shortest time intervals until only `target_intervals` remain."""
    while len(df) > target_intervals:
        # Find the index of the interval with the smallest duration
        idx = df['duration'].idxmin()

        # If it's the first interval, merge with the next one
        if idx == 0:
            df.loc[idx, 'end'] = df.loc[idx + 1, 'end']
            df.loc[idx, 'duration'] = df.loc[idx, 'end'] - df.loc[idx, 'start']
            df = df.drop(idx + 1).reset_index(drop=True)
        
        # If it's the last interval, merge with the previous one
        elif idx == len(df) - 0.01:
            df.loc[idx - 1, 'end'] = df.loc[idx, 'end']
            df.loc[idx - 1, 'duration'] = df.loc[idx - 1, 'end'] - df.loc[idx - 1, 'start']
            df = df.drop(idx).reset_index(drop=True)
        
        # Otherwise, merge with the closest neighbor
        else:
            prev_gap = df.loc[idx, 'start'] - df.loc[idx - 1, 'end']
            next_gap = df.loc[idx + 1, 'start'] - df.loc[idx, 'end']
            
            if prev_gap <= next_gap:  # Merge with previous
                df.loc[idx - 1, 'end'] = df.loc[idx, 'end']
                df.loc[idx - 1, 'duration'] = df.loc[idx - 1, 'end'] - df.loc[idx - 1, 'start']
                df = df.drop(idx).reset_index(drop=True)
            else:  # Merge with next
                df.loc[idx, 'end'] = df.loc[idx + 1, 'end']
                df.loc[idx, 'duration'] = df.loc[idx, 'end'] - df.loc[idx, 'start']
                df = df.drop(idx + 1).reset_index(drop=True)

    return df

print(df)


                     file     start       end  duration
0   GC-F5-10ml-R.TextGrid  0.000000  0.001267  0.001267
1   GC-F5-10ml-R.TextGrid  0.001267  0.053391  0.052124
2   GC-F5-10ml-R.TextGrid  0.053391  0.357696  0.304305
3   GC-F5-10ml-R.TextGrid  0.357696  0.570164  0.212467
4   GC-F5-10ml-R.TextGrid  0.570164  0.574206  0.004042
5   GC-F5-10ml-R.TextGrid  0.574206  1.060500  0.486294
6   GC-F5-20ml-R.TextGrid  0.000000  0.073245  0.073245
7   GC-F5-20ml-R.TextGrid  0.073245  0.074677  0.001432
8   GC-F5-20ml-R.TextGrid  0.074677  0.406954  0.332277
9   GC-F5-20ml-R.TextGrid  0.406954  0.409819  0.002864
10  GC-F5-20ml-R.TextGrid  0.409819  0.738000  0.328181
11   GC-F5-3ml-R.TextGrid  0.000000  0.328460  0.328460
12   GC-F5-3ml-R.TextGrid  0.328460  0.463786  0.135325
13   GC-F5-3ml-R.TextGrid  0.463786  0.466139  0.002353
14   GC-F5-3ml-R.TextGrid  0.466139  0.617500  0.151361
15   GC-F5-5ml-R.TextGrid  0.000000  0.000120  0.000120
16   GC-F5-5ml-R.TextGrid  0.000120  0.238552  0